# Belgian Urban Heat Monitoring Intelligence

> **AI-assisted urban heat *anomaly* detection using satellite imagery, citizen science sensors and official meteorological baselines.**

This notebook models **heat anomalies** — how much hotter or cooler a location is vs. the city daily mean — rather than absolute temperature. This is the correct approach to reveal the urban heat island (UHI) effect:
- Parks, water, trees → **negative anomaly** (blue on maps) — vegetation cools via evapotranspiration
- Pavements, rooftops, dense urban → **positive anomaly** (red on maps) — impervious surfaces absorb and re-emit heat

**⚠ Geographic scope:** Sentinel-2 imagery was downloaded specifically for **Leuven, Belgium** only. The spatial prediction grid therefore covers the Leuven AOI (50.84–50.94°N, 4.63–4.78°E). The RMI AWS integration extends the analysis to 5 Belgian cities for temporal baseline comparison, but the satellite-derived UHI maps are Leuven-specific.

**Data sources:**
- **Leuven.cool** citizen science network · ~155 stations · Q3 (Jul–Sep) 2023–2025  
  *Zenodo DOI: 10.5281/zenodo.14893734*
- **Sentinel-2 L2A** imagery · 7 low-cloud summer scenes · downloaded via  
  *Copernicus Data Space Browser: https://browser.dataspace.copernicus.eu* · Bands: B04, B08, B11
- **RMI AWS** official network · 17 Belgian stations · daily observations via  
  *opendata.meteo.be / openkmi Python package*
- **ESA WorldCover 10 m** land use map · Belgium 2021 v2 · visualised via  
  *Terrascope viewer: https://viewer.terrascope.be*

**Models:** Random Forest · XGBoost (+ tuned) · CNN (image patches)  
**Target:** `temp_anomaly` = station temp − city daily mean

---

## 2. Datasets

### 2.1 Leuven.cool Citizen Science Sensor Network

**What it is:**  
Leuven.cool is a network of approximately 155 low-cost personal weather stations installed across Leuven and operated by residents, schools, and the city. The stations are Fine Offset WH2600 units. The network was created as part of the Leuven.cool citizen science initiative to study the urban microclimate.

**Source:** Zenodo DOI: `10.5281/zenodo.14893734`  
**Files used:** `RAWDATA2023Q3.csv`, `RAWDATA2024Q3.csv`, `RAWDATA2025Q3.csv`, `STATIONS.csv`  
**Coverage:** Approximately 100 km² of the Leuven area, with around 800 m average spacing between stations  
**Period:** Q3, meaning July–September, for 2023, 2024, and 2025  
**Raw size:** 30,894,890 rows loaded across three years  

#### RAWDATA column definitions

| Column | Unit | Definition |
|---|---:|---|
| `OBSID` | — | Unique identifier for each observation. Not used as a feature. |
| `ID` | — | Station identifier. Used to join with `STATIONS.csv`. |
| `TEMPF` | °F | Air temperature measured by the station. Converted to °C for modelling. |
| `HUMIDITY` | % | Relative humidity. Used as a model feature. |
| `DEWPTF` | °F | Dew point temperature. Converted to °C. |
| `WINDCHILLF` | °F | Wind chill index. Not used because it is derived from temperature and wind speed. |
| `WINDDIR` | degrees | Wind direction from 0–360°. Dropped because it requires circular encoding and had limited value for this prototype. |
| `WINDSPEEDMPH` | mph | Wind speed. Converted to m/s and used as a model feature. |
| `WINDGUSTMPH` | mph | Maximum wind gust speed. Converted to m/s, but dropped because it is strongly correlated with `WINDSPEEDMPH`. |
| `RAININ` | inch/hour | Rain intensity at measurement time. Dropped because rainfall was near-zero for most Q3 summer records. |
| `DAILYRAININ` | inch | Total daily rainfall. Dropped for the same reason as `RAININ`. |
| `SOLARRADIATION` | W/m² | Incoming solar radiation measured at the station. Used as a model feature. |
| `UV` | index | UV index. Dropped because it is highly correlated with `SOLARRADIATION` with correlation above 0.95. |
| `BAROMIN` | inHg | Atmospheric pressure. Dropped because pressure does not directly explain local UHI anomalies. |
| `DATEUTC` | UTC timestamp | Server reception time of the measurement. Used for temporal alignment with satellite scenes. |
| `DATEUTC_STAT` | UTC timestamp | Timestamp reported by the station itself. Dropped because `DATEUTC` was used instead. |

#### STATIONS.csv column definitions

| Column | Definition |
|---|---|
| `ID` | Station identifier. Matches the `ID` column in the raw data files. |
| `WOWID` | Corresponding identifier in the UK Met Office Weather Observations Website. Not used. |
| `LATITUDE` | Latitude of the station in decimal degrees, using WGS84. |
| `LONGITUDE` | Longitude of the station in decimal degrees, using WGS84. |
| `ALTITUDE` | Altitude of the station above sea level in metres. Used as a model feature. |

#### Why Q3 only?

Urban heat islands are most visible during summer, when solar radiation is highest. Q3 also matches the available Sentinel-2 scenes, which are mostly from July and August.

#### Why only 5,575 rows end up in the model?

The raw files contain 30.9 million rows, but the model only uses observations that can be matched with a Sentinel-2 satellite scene. After filtering observations within ±7 days of each satellite scene, aggregating to daily station-level records, and joining with satellite features, 5,575 station-day rows remain.

---

### 2.2 Sentinel-2 L2A Satellite Imagery

**What it is:**  
Sentinel-2 is a European Space Agency satellite mission that captures multispectral images of the Earth's surface. The L2A product contains surface reflectance, meaning the fraction of sunlight reflected by the ground after atmospheric correction. The raw digital numbers are scaled from 0 to 10,000 and divided by 10,000 to convert them to reflectance values between 0 and 1.

**Source:** ESA Copernicus Data Space Browser  
**Website:** `browser.dataspace.copernicus.eu`  
**Tile:** `31UES`, covering Leuven and the surrounding area  
**Selection criteria:** Cloud cover ≤ 10%, Q3 summer dates from 2023 to 2025  
**Format:** JP2 files per band, stored as `sentinel2/YYYY-MM-DD/BXX.jp2`  
**Geographic scope:** Leuven only  

#### Scene inventory

| Date | Year | Notes |
|---|---:|---|
| 2023-08-10 | 2023 | Training scene |
| 2023-08-20 | 2023 | Representative scene for 2023 |
| 2023-08-23 | 2023 | Training scene |
| 2024-07-30 | 2024 | Only available scene for 2024 |
| 2025-07-02 | 2025 | Training scene |
| 2025-08-11 | 2025 | Representative scene for 2025 |
| 2025-08-12 | 2025 | Training scene |

Only one Sentinel-2 scene was used for 2024 because no other summer acquisition over Leuven met the cloud cover threshold. This was a real data limitation rather than a deliberate choice.

#### Band definitions

| Band | Name | Wavelength | Resolution | Used | Reason |
|---|---|---:|---:|:---:|---|
| B04 | Red | 665 nm | 10 m | ✓ | Needed for NDVI. Vegetation absorbs strongly in the red band. |
| B08 | NIR | 842 nm | 10 m | ✓ | Needed for NDVI. Vegetation reflects strongly in near-infrared. |
| B11 | SWIR-1 | 1610 nm | 20 m | ✓ | Needed for NDBI. Dry concrete and asphalt reflect strongly in SWIR. |
| B12 | SWIR-2 | 2190 nm | 20 m | ✗ | Dropped because it is redundant with B11 for this prototype and may add noise. |

#### Derived indices

| Index | Formula | Range | High value means | UHI link |
|---|---|---:|---|---|
| NDVI | `(B08 − B04) / (B08 + B04)` | −1 to +1 | Dense, healthy vegetation | Vegetation cools through evapotranspiration, so high NDVI is expected to lower heat anomaly. |
| NDBI | `(B11 − B08) / (B11 + B08)` | −1 to +1 | Built-up or impervious surface | Concrete and asphalt absorb and retain heat, so high NDBI is expected to increase heat anomaly. |

In Leuven, NDVI is typically higher around parks, tree corridors, the Dijle valley, and university green spaces. In the dense city centre, NDVI is much lower. NDBI is highest over industrial zones, dense rooftop areas, and built-up blocks, and lowest over parks and water.

#### Note on B11 resolution

B04 and B08 are available at 10 m resolution, while B11 is available at 20 m resolution. For point sampling at station GPS coordinates, this is acceptable. For CNN image patches, however, the B11 patch covers four times the physical area of the B04 and B08 patches. This is a known limitation of the CNN experiment.

---

### 2.3 RMI AWS — Royal Meteorological Institute Automatic Weather Stations

**What it is:**  
The Royal Meteorological Institute of Belgium operates a network of official automatic weather stations across the country. These are WMO-standard installations that are regularly maintained and calibrated. The data is publicly available through the RMI Open Data portal and can be accessed using the `openkmi` Python package.

**Source:** `opendata.meteo.be`  
**Metadata:** `RMI_DATASET_AWS_10MIN`  
**Access:** `pip install openkmi`  
**Licence:** Creative Commons Attribution 4.0, or CC BY 4.0  
**Temporal extent:** 1995 to present, with 10-minute raw observations  
**Used at:** Daily frequency with `freq='D'`  
**Loaded rows:** 1,104 daily rows, corresponding to 5 cities × approximately 92 summer days × 3 years  

#### Stations used

| City | Station name | Station ID |
|---|---|---:|
| Leuven, nearest station | Diest | 6438 |
| Brussels | Uccle | 6447 |
| Ghent | Gent-Melle | 6407 |
| Antwerp | Antwerpen | 6414 |
| Liège | Bierset | 6459 |

#### Column definitions from the daily API

| API column name | Renamed to | Unit | Definition |
|---|---|---:|---|
| `temp_avg` | `temp_c` | °C | Daily mean air temperature measured in a radiation shelter. |
| `humidity_rel_shelter_avg` | `humidity` | % | Daily mean relative humidity in shelter. |
| `short_wave_from_sky_avg` | `solar_rad` | W/m² | Daily mean incoming shortwave solar radiation. |

#### Important API note

The daily API with `freq='D'` uses lowercase snake_case column names. The 10-minute API uses column names such as `TEMP_DRY_SHELTER_AVG` and `HUMIDITY_REL_SHELTER_AVG`. These are different endpoints with different schemas. Passing 10-minute column names to the daily endpoint causes a runtime error. This bug was fixed in the current version.

#### Why RMI is needed

Leuven.cool sensors are citizen-operated and may contain siting bias, calibration drift, or local installation effects. RMI data provides a calibrated reference from official stations. It is used to:

1. Cross-check Leuven.cool readings.
2. Compare year-over-year summer temperature trends across Belgian cities.
3. Place Leuven's urban heat patterns in a broader national context.

---

## Table of Contents
1. [Setup & Config](#1)
2. [Phase 1 — Data Collection & Provenance](#2)
   - 2a. Sentinel-2 L2A — why, bands, geographic scope, scenes
   - 2b. Leuven.cool — why, columns used/dropped
   - 2c. RMI AWS — why, station codes, columns
   - 2d. ESA WorldCover — land use context
   - 2e. Date-filtered loading strategy
3. [Phase 2 — Data Quality Assessment](#3)
4. [Phase 3 — Preprocessing & Anomaly Target](#4)
   - What is a heat anomaly?
   - What causes hotspots vs cool zones?
   - Preprocessing steps
5. [Phase 4 — RMI Baseline Integration](#5)
   - Why official data is needed
   - API column name correction
6. [Phase 5 — Feature Engineering](#6)
   - NDVI / NDBI derivation and rationale
   - WorldCover validation
   - Full feature table
7. [Phase 6 — ML Modeling](#7)
   - Why three models
   - Train / Test / Retrain strategy
   - 7a. 80/20 Split + 5-Fold CV + LOSOCV
   - 7b. Random Forest
   - 7c. XGBoost
   - 7d. CNN — architecture + underfit explanation
   - 7e. Hyperparameter Tuning
8. [Phase 7 — Evaluation & Model Comparison](#8)
   - Why R² is moderate — explained
   - Standard CV vs LOSOCV
   - RMI Belgian city findings
9. [Phase 8 — Heat Anomaly Maps](#9)
   - Flat grid explanation
   - Multi-year comparison
   - RMI city conclusion
10. [Phase 9 — Deployment Notes](#10)

---

## 1. Setup & Config

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (KFold, cross_val_score,
                                     RandomizedSearchCV, StratifiedShuffleSplit)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split as tts
import xgboost as xgb

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

print('All imports OK')